In [21]:
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from src.features import clean_titanic, engineer_features, get_feature_matrix

In [10]:
# Prepare data
df = sns.load_dataset('titanic')
df = clean_titanic(df)
df = engineer_features(df)
x, y = get_feature_matrix(df)

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

In [11]:
# Pipeline = preprocessing + model in one object
# CRITICAL: scalar.fit() only on train data (pipeline handles this)

pipe_lr = Pipeline([
    ('scalar', StandardScaler()),
    ('model', LogisticRegression(max_iter=500))
])

pipe_rf = Pipeline([
    ('scalar', StandardScaler()),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42))
])

pipe_dtc = Pipeline([
    ('scalar', StandardScaler()),
    ('model', DecisionTreeClassifier(max_depth=5, random_state=42))
])

In [12]:
# Train and compare 
models = {'Logistic Regression': pipe_lr, 'Random Forest':pipe_rf, 'Decision Tree Classifier': pipe_dtc}

for name, pipe in models.items():
    pipe.fit(x_train, y_train)
    train_score = pipe.score(x_train, y_train)
    test_score = pipe.score(x_test, y_test)
    cv_scores = cross_val_score(pipe, x_train, y_train, cv=5)
    print(f"{name}:")
    print(f" Train accuracy: {train_score:.3f}")
    print(f" Test accuracy: {test_score:.3f}")
    print(f" CV mean ± std: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
    print()

Logistic Regression:
 Train accuracy: 0.805
 Test accuracy: 0.804
 CV mean ± std: 0.796 ± 0.018

Random Forest:
 Train accuracy: 0.983
 Test accuracy: 0.788
 CV mean ± std: 0.795 ± 0.038

Decision Tree Classifier:
 Train accuracy: 0.875
 Test accuracy: 0.765
 CV mean ± std: 0.823 ± 0.035



In [17]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score

# Grid Search: try multiple hyperparameter combos
param_grid = {
    'model__n_estimators': [50, 100, 200],
    'model__max_depth': [3, 5, 10, None],
    'model__min_samples_split': [2, 5, 10],
}

In [18]:
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42))
])

In [19]:
grid_search = GridSearchCV(
    pipe, param_grid, cv=5, scoring='f1', n_jobs=-1, verbose=1
)
grid_search.fit(x_train, y_train)

print(f"Best params: {grid_search.best_params_}")
print(f"Best CV F1: {grid_search.best_score_:.3f}")
print(f"Test F1: {f1_score(y_test, grid_search.predict(x_test)):.3f}")

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best params: {'model__max_depth': 10, 'model__min_samples_split': 10, 'model__n_estimators': 50}
Best CV F1: 0.779
Test F1: 0.712


In [20]:
# Use best model going forward
best_model = grid_search.best_estimator_